# Voice Deepfake Detection with SE-ResNet-18 and Log-Mel Spectrograms
## ASVspoof 2019 Logical Access Benchmark: Full Tri-Partition Evaluation (Train, Dev, and Eval)

### Abstract and Scientific Background
Synthetic speech generation techniques (including neural text-to-speech vocoders and acoustic voice conversion models) induce unnatural spectro-temporal artifacts, harmonic phase discontinuities, and high-frequency spectral rolloff discrepancies. While shallow baseline classifiers frequently achieve low error rates on known attacks seen during training, they suffer severe performance degradation when confronted with unseen generative architectures.

This study formulates an end-to-end deep learning framework based on **SE-ResNet-18 (Squeeze-and-Excitation Residual Network)** combined with 128-channel standardized Log-Mel spectrograms. The core technical contributions include:
1. **Acoustic Engineering Pipeline**: Pre-emphasis high-frequency boosting ($y[t] = x[t] - 0.97 x[t-1]$), Voice Activity Detection (VAD) silent trimming, length normalization to a 4.00-second window (64,000 samples), and 128-channel Log-Mel extraction ($N_{\text{fft}}=1024, \text{hop}=256$) standardized to zero mean and unit variance.
2. **Squeeze-and-Excitation Channel Attention**: SE blocks with reduction factor $r=8$ dynamically model inter-dependencies between spectral channels, selectively amplifying artifact-carrying frequency bands while suppressing invariant vocal tract resonances:
$$\mathbf{s} = \sigma(\mathbf{W}_2 \cdot \text{ReLU}(\mathbf{W}_1 \cdot \mathbf{z}))$$
3. **Focal Loss with Label Smoothing**: Directly addresses the severe 1:8.7 class imbalance between authentic speech and spoofed utterances while penalizing borderline synthetic samples:
$$\mathcal{L}_{\text{Focal}} = -\alpha_t (1 - p_t)^\gamma \log(p_t)$$
4. **Complete Tri-Partition Evaluation**: Eliminates the closed-world limitation of prior experiments by performing exhaustive batch evaluation across all **71,237 utterances of the official ASVspoof 2019 Evaluation partition**, rigorously auditing detection accuracy across eleven unseen out-of-distribution attack mechanisms (**A07 through A19**).

### Pipeline Roadmap
- Step 1: Hardware diagnostics and deterministic environment setup
- Step 2: Multi-path dataset discovery across Kaggle input directories
- Step 3: Unified protocol parsing across Train (25,380), Dev (24,844), and Eval (71,237)
- Step 4: Speaker independence audit across partitions
- Steps 5-11: Comprehensive Exploratory Data Analysis (EDA) producing 7 publication figures
- Step 12: 2D SpecAugment time-frequency regularization
- Step 13: PyTorch Dataset with on-the-fly Mel extraction and WeightedRandomSampler
- Step 14: SE-ResNet-18 architecture formulation with SE attention and latent projection
- Step 15: Focal Loss optimization with Cosine Annealing scheduler
- Step 16: Biometric evaluation engine (EER, normalized min t-DCF, ROC, DET, PR)
- Step 17: Full 20-epoch training and validation tracking with real-time Dev EER
- Step 18: Training loss convergence and validation trajectory diagnostics
- Step 19: Full-scale Evaluation Partition batch inference over all 71,237 utterances
- Steps 20-22: Biometric performance curves (ROC, DET, PR on Dev and Eval)
- Step 23: Normalized Confusion Matrix on the Evaluation partition
- Step 24: Granular 19-attack vulnerability breakdown (A01 through A19)
- Step 25: 2D t-SNE latent representation clustering (Bonafide vs Known vs Unseen spoofs)
- Step 26: Spectro-temporal explainability via Grad-CAM saliency heatmaps
- Step 27: Live single-file inference verification
- Step 28: Final artifact serialization and executive benchmark summary


In [ ]:
import os
import sys
import time
import math
import json
import random
import warnings
import numpy as np
import pandas as pd
import soundfile as sf
import librosa
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_curve, roc_auc_score, confusion_matrix, precision_recall_curve, average_precision_score
from sklearn.manifold import TSNE

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

warnings.filterwarnings("ignore")

seed_val = 42
random.seed(seed_val)
np.random.seed(seed_val)
torch.manual_seed(seed_val)
if torch.cuda.is_available():
    torch.cuda.manual_seed(seed_val)
    torch.cuda.manual_seed_all(seed_val)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_amp = torch.cuda.is_available()

print("Hardware and Runtime Diagnostics:")
print(f"  PyTorch Version:  {torch.__version__}")
print(f"  Librosa Version:  {librosa.__version__}")
print(f"  Compute Device:   {device}")
if torch.cuda.is_available():
    print(f"  GPU Identifier:   {torch.cuda.get_device_name(0)}")
    print(f"  VRAM Allocated:   {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
    print("  Mixed Precision:  Enabled (torch.amp.autocast)")
else:
    print("  WARNING: GPU accelerator not detected. Execution will proceed on CPU.")

fig_dir = "/kaggle/working/figures"
weights_dir = "/kaggle/working/models"
os.makedirs(fig_dir, exist_ok=True)
os.makedirs(weights_dir, exist_ok=True)
print(f"Output Figure Directory:  {fig_dir}")
print(f"Output Model Directory:   {weights_dir}")


In [ ]:
def locate_dataset():
    search_roots = ["/kaggle/input", "/kaggle/working", "."]
    candidates = [
        "/kaggle/input/datasets/awsaf49/asvpoof-2019-dataset/LA/LA",
        "/kaggle/input/datasets/awsaf49/asvpoof-2019-dataset/LA",
        "/kaggle/input/datasets/awsaf49/asvpoof-2019-dataset",
        "/kaggle/input/asvpoof-2019-dataset/LA/LA",
        "/kaggle/input/asvpoof-2019-dataset/LA",
        "/kaggle/input/asvpoof-2019-dataset",
        "/kaggle/input/asvspoof-2019-dataset/LA/LA",
        "/kaggle/input/asvspoof-2019-dataset/LA",
        "/kaggle/input/asvspoof-2019-dataset",
        "/kaggle/input/asvspoof-2019/LA/LA",
        "/kaggle/input/asvspoof-2019/LA",
        "/kaggle/input/asvspoof-2019",
        "/kaggle/input/asvspoof2019/LA/LA",
        "/kaggle/input/asvspoof2019/LA",
        "/kaggle/input/asvspoof2019"
    ]
    protocols = {"train": None, "dev": None, "eval": None}
    audio_dirs = {"train": None, "dev": None, "eval": None}

    for c in candidates:
        if os.path.isdir(c):
            for part, sfx in [("train", "trn"), ("dev", "trl"), ("eval", "trl")]:
                proto_p = os.path.join(c, "ASVspoof2019_LA_cm_protocols", f"ASVspoof2019.LA.cm.{part}.{sfx}.txt")
                if os.path.isfile(proto_p) and not protocols[part]:
                    protocols[part] = proto_p
                flac_p = os.path.join(c, f"ASVspoof2019_LA_{part}", "flac")
                if os.path.isdir(flac_p) and not audio_dirs[part]:
                    audio_dirs[part] = flac_p
                elif os.path.isdir(os.path.join(c, f"ASVspoof2019_LA_{part}")) and not audio_dirs[part]:
                    audio_dirs[part] = os.path.join(c, f"ASVspoof2019_LA_{part}")

    if any(v is None for v in list(protocols.values()) + list(audio_dirs.values())):
        for s_root in search_roots:
            if not os.path.exists(s_root):
                continue
            for root, dirs, files in os.walk(s_root, followlinks=True):
                for f in files:
                    fl = f.lower()
                    if "cm" in fl and fl.endswith(".txt") and not f.startswith("._"):
                        if "train" in fl and ("trn" in fl or "train" in fl) and not protocols["train"]:
                            protocols["train"] = os.path.join(root, f)
                        elif "dev" in fl and ("trl" in fl or "dev" in fl) and not protocols["dev"]:
                            protocols["dev"] = os.path.join(root, f)
                        elif "eval" in fl and ("trl" in fl or "eval" in fl) and not protocols["eval"]:
                            protocols["eval"] = os.path.join(root, f)
                for d in list(dirs):
                    dl = d.lower()
                    for part in ["train", "dev", "eval"]:
                        if (f"la_{part}" in dl or f"la.{part}" in dl or f"_{part}" in dl) and not audio_dirs[part]:
                            sub_flac = os.path.join(root, d, "flac")
                            if os.path.isdir(sub_flac):
                                audio_dirs[part] = sub_flac
                            elif os.path.isdir(os.path.join(root, d)):
                                audio_dirs[part] = os.path.join(root, d)
                if "flac" in dirs:
                    dirs.remove("flac")

    return protocols, audio_dirs

protocols, audio_dirs = locate_dataset()

print("Resolved Dataset Partitions:")
for p in ["train", "dev", "eval"]:
    proto_stat = "FOUND" if protocols[p] and os.path.isfile(protocols[p]) else "MISSING"
    audio_stat = "FOUND" if audio_dirs[p] and os.path.isdir(audio_dirs[p]) else "MISSING"
    n_files = len(os.listdir(audio_dirs[p])) if audio_stat == "FOUND" else 0
    print(f"  [{p.upper()}]")
    print(f"    Protocol:  {proto_stat} -> {protocols[p]}")
    print(f"    Audio Dir: {audio_stat} -> {audio_dirs[p]} ({n_files:,} files)")

if not protocols["train"] or not os.path.isfile(protocols["train"]):
    raise FileNotFoundError("Critical: Train protocol file missing. Ensure ASVspoof 2019 dataset is attached to Kaggle notebook.")
if not audio_dirs["train"] or not os.path.isdir(audio_dirs["train"]):
    raise FileNotFoundError("Critical: Train audio directory missing.")


In [ ]:
parsed_records = []

for partition in ["train", "dev", "eval"]:
    proto_path = protocols.get(partition)
    audio_path = audio_dirs.get(partition)
    if not proto_path or not os.path.isfile(proto_path):
        print(f"Warning: Partition {partition} protocol file not accessible.")
        continue
    if not audio_path or not os.path.isdir(audio_path):
        print(f"Warning: Partition {partition} audio directory not accessible.")
        continue

    with open(proto_path, "r", encoding="utf-8") as f:
        for line in f:
            tokens = line.strip().split()
            if len(tokens) < 5:
                continue
            spk_id = tokens[0]
            audio_id = tokens[1]
            env_id = tokens[2]
            atk_token = tokens[3]
            key_token = tokens[4].lower()

            atk_id = "Bonafide" if (key_token == "bonafide" or atk_token == "-") else atk_token
            is_spoof = 1 if key_token == "spoof" else 0

            file_full_path = os.path.join(audio_path, f"{audio_id}.flac")
            parsed_records.append({
                "speaker_id": spk_id,
                "audio_id": audio_id,
                "environment_id": env_id,
                "attack_id": atk_id,
                "key": key_token,
                "is_spoof": is_spoof,
                "partition": partition,
                "file_path": file_full_path
            })

manifest_df = pd.DataFrame(parsed_records)
print(f"Total Database Utterances Parsed: {len(manifest_df):,}")

partition_summary = []
for p in ["train", "dev", "eval"]:
    sub = manifest_df[manifest_df["partition"] == p]
    if len(sub) == 0:
        continue
    bon_cnt = int((sub["key"] == "bonafide").sum())
    spf_cnt = int((sub["key"] == "spoof").sum())
    tot_cnt = len(sub)
    ratio_str = f"{spf_cnt / max(bon_cnt, 1):.2f}:1"
    attacks_present = sorted([a for a in sub["attack_id"].unique() if a != "Bonafide"])
    partition_summary.append({
        "Partition": p.upper(),
        "Total Utterances": f"{tot_cnt:,}",
        "Bonafide": f"{bon_cnt:,}",
        "Spoof": f"{spf_cnt:,}",
        "Spoof:Bonafide Ratio": ratio_str,
        "Unique Speakers": sub["speaker_id"].nunique(),
        "Attack IDs": ", ".join(attacks_present)
    })

summary_display = pd.DataFrame(partition_summary)
print(summary_display.to_string(index=False))

train_df = manifest_df[manifest_df["partition"] == "train"].reset_index(drop=True)
dev_df = manifest_df[manifest_df["partition"] == "dev"].reset_index(drop=True)
eval_df = manifest_df[manifest_df["partition"] == "eval"].reset_index(drop=True)

print("")
print("Partition Split Complete:")
print(f"  Train: {len(train_df):,} utterances (A01-A06)")
print(f"  Dev:   {len(dev_df):,} utterances (A01-A06)")
print(f"  Eval:  {len(eval_df):,} utterances (A07-A19 unseen attacks)")


In [ ]:
spks_train = set(train_df["speaker_id"].unique())
spks_dev = set(dev_df["speaker_id"].unique())
spks_eval = set(eval_df["speaker_id"].unique()) if len(eval_df) > 0 else set()

ov_train_dev = spks_train.intersection(spks_dev)
ov_train_eval = spks_train.intersection(spks_eval)
ov_dev_eval = spks_dev.intersection(spks_eval)

print("Speaker Independence Audit:")
print(f"  Train Unique Speakers: {len(spks_train)}")
print(f"  Dev Unique Speakers:   {len(spks_dev)}")
print(f"  Eval Unique Speakers:  {len(spks_eval)}")
print(f"  Overlap (Train & Dev):  {len(ov_train_dev)} (Expected: 0)")
print(f"  Overlap (Train & Eval): {len(ov_train_eval)} (Expected: 0)")
print(f"  Overlap (Dev & Eval):   {len(ov_dev_eval)} (Expected: 0)")

assert len(ov_train_dev) == 0, "Data leakage detected: Speaker overlap between Train and Dev partitions."
assert len(ov_train_eval) == 0, "Data leakage detected: Speaker overlap between Train and Eval partitions."
print("Speaker independence strictly verified across all three experimental partitions.")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

parts = [p for p in ["train", "dev", "eval"] if len(manifest_df[manifest_df["partition"] == p]) > 0]
bon_counts = [int(((manifest_df["partition"] == p) & (manifest_df["key"] == "bonafide")).sum()) for p in parts]
spf_counts = [int(((manifest_df["partition"] == p) & (manifest_df["key"] == "spoof")).sum()) for p in parts]

x_idx = np.arange(len(parts))
b_width = 0.35

axes[0].bar(x_idx - b_width/2, bon_counts, b_width, label="Authentic (Bonafide)", color="steelblue")
axes[0].bar(x_idx + b_width/2, spf_counts, b_width, label="Synthetic (Spoof)", color="firebrick")
axes[0].set_title("Class Utterance Breakdown Across Partitions", fontsize=12)
axes[0].set_xlabel("Dataset Partition", fontsize=11)
axes[0].set_ylabel("Total Number of Utterances", fontsize=11)
axes[0].set_xticks(x_idx)
axes[0].set_xticklabels([p.upper() for p in parts], fontsize=10)
axes[0].legend(fontsize=10)
axes[0].grid(axis="y", linestyle="--", alpha=0.4)

for i in range(len(parts)):
    axes[0].text(x_idx[i] - b_width/2, bon_counts[i] + max(spf_counts)*0.015, f"{bon_counts[i]:,}", ha="center", fontsize=9)
    axes[0].text(x_idx[i] + b_width/2, spf_counts[i] + max(spf_counts)*0.015, f"{spf_counts[i]:,}", ha="center", fontsize=9)

ratios = [spf_counts[i] / max(bon_counts[i], 1) for i in range(len(parts))]
axes[1].bar([p.upper() for p in parts], ratios, color="darkslateblue", width=0.4)
axes[1].set_title("Class Imbalance Ratio (Spoof : Bonafide)", fontsize=12)
axes[1].set_xlabel("Dataset Partition", fontsize=11)
axes[1].set_ylabel("Imbalance Ratio", fontsize=11)
axes[1].grid(axis="y", linestyle="--", alpha=0.4)

for i, r in enumerate(ratios):
    axes[1].text(i, r + 0.15, f"{r:.2f}:1", ha="center", fontsize=10, fontweight="bold")

plt.tight_layout()
plt.savefig(os.path.join(fig_dir, "01_class_distribution_breakdown.png"), dpi=300, bbox_inches="tight")
plt.show()
print("Saved Figure 01: Class distribution breakdown.")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

train_dev_atks = manifest_df[manifest_df["partition"].isin(["train", "dev"]) & (manifest_df["attack_id"] != "Bonafide")]
td_counts = train_dev_atks["attack_id"].value_counts().sort_index()

eval_atks = manifest_df[(manifest_df["partition"] == "eval") & (manifest_df["attack_id"] != "Bonafide")]
eval_counts = eval_atks["attack_id"].value_counts().sort_index()

axes[0].bar(td_counts.index, td_counts.values, color="coral", edgecolor="black", width=0.55)
axes[0].set_title("Known Attack Distribution: Train & Dev (A01 - A06)", fontsize=12)
axes[0].set_xlabel("Spoofing Attack Algorithm ID", fontsize=11)
axes[0].set_ylabel("Number of Utterances", fontsize=11)
axes[0].grid(axis="y", linestyle="--", alpha=0.4)
for idx, val in enumerate(td_counts.values):
    axes[0].text(idx, val + max(td_counts.values)*0.015, f"{val:,}", ha="center", fontsize=9)

axes[1].bar(eval_counts.index, eval_counts.values, color="mediumpurple", edgecolor="black", width=0.6)
axes[1].set_title("Unseen Out-of-Distribution Attacks: Evaluation (A07 - A19)", fontsize=12)
axes[1].set_xlabel("Spoofing Attack Algorithm ID", fontsize=11)
axes[1].set_ylabel("Number of Utterances", fontsize=11)
axes[1].grid(axis="y", linestyle="--", alpha=0.4)
for idx, val in enumerate(eval_counts.values):
    axes[1].text(idx, val + max(eval_counts.values)*0.015, f"{val:,}", ha="center", fontsize=8, rotation=45)

plt.tight_layout()
plt.savefig(os.path.join(fig_dir, "02_attack_distribution_analysis.png"), dpi=300, bbox_inches="tight")
plt.show()
print("Saved Figure 02: Attack distribution analysis.")


In [ ]:
sample_rows = manifest_df.sample(min(200, len(manifest_df)), random_state=42)
durations = []
sample_rates = []

for _, r in sample_rows.iterrows():
    fpath = r["file_path"]
    if os.path.isfile(fpath):
        try:
            info = sf.info(fpath)
            durations.append(info.duration)
            sample_rates.append(info.samplerate)
        except Exception:
            pass

durations = np.array(durations)
sample_rates = np.array(sample_rates)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

axes[0].hist(durations, bins=25, color="teal", edgecolor="black", alpha=0.8)
axes[0].axvline(np.mean(durations), color="red", linestyle="--", lw=2, label=f"Mean: {np.mean(durations):.2f}s")
axes[0].axvline(4.0, color="orange", linestyle=":", lw=2, label="Fixed Target Window: 4.00s")
axes[0].set_title("Speech Duration Distribution (Audit Sample N=200)", fontsize=12)
axes[0].set_xlabel("Audio Duration (Seconds)", fontsize=11)
axes[0].set_ylabel("Utterance Count", fontsize=11)
axes[0].legend(fontsize=10)
axes[0].grid(True, linestyle="--", alpha=0.3)

sr_counts = pd.Series(sample_rates).value_counts()
axes[1].bar([f"{int(sr)} Hz" for sr in sr_counts.index], sr_counts.values, color="steelblue", width=0.3)
axes[1].set_title("Sample Rate Consistency Audit", fontsize=12)
axes[1].set_xlabel("Sampling Frequency", fontsize=11)
axes[1].set_ylabel("Utterance Count", fontsize=11)
axes[1].grid(axis="y", linestyle="--", alpha=0.3)
for i, v in enumerate(sr_counts.values):
    axes[1].text(i, v + 2, f"{v} (100%)", ha="center", fontsize=10, fontweight="bold")

plt.tight_layout()
plt.savefig(os.path.join(fig_dir, "03_audio_duration_length_distribution.png"), dpi=300, bbox_inches="tight")
plt.show()
print("Saved Figure 03: Audio duration and sample rate audit.")


In [ ]:
def load_raw_audio(path, target_sr=16000):
    try:
        y, orig_sr = sf.read(path, dtype="float32")
        if y.ndim > 1:
            y = np.mean(y, axis=1)
        if orig_sr != target_sr:
            y = librosa.resample(y, orig_sr=orig_sr, target_sr=target_sr)
        return y
    except Exception:
        return np.zeros(64000, dtype=np.float32)

def preprocess_audio(y, is_train=False, target_len=64000, alpha=0.97, top_db=40):
    if len(y) < 100:
        return np.zeros(target_len, dtype=np.float32)
    y_filt = np.concatenate([[y[0]], y[1:] - alpha * y[:-1]])
    try:
        intervals = librosa.effects.split(y=y_filt, top_db=top_db)
        if len(intervals):
            trimmed = np.concatenate([y_filt[s:e] for s, e in intervals])
            if len(trimmed) > 1000:
                y_filt = trimmed
    except Exception:
        pass
    n_samples = len(y_filt)
    if n_samples >= target_len:
        start = np.random.randint(0, n_samples - target_len + 1) if is_train else (n_samples - target_len) // 2
        y_proc = y_filt[start:start + target_len]
    else:
        y_proc = np.pad(y_filt, (0, target_len - n_samples), mode="wrap")
    peak = np.max(np.abs(y_proc))
    if peak > 1e-6:
        y_proc = y_proc / peak
    return y_proc.astype(np.float32)

def extract_log_mel(y, sr=16000, n_fft=1024, hop_length=256, n_mels=128, fmin=20, fmax=8000, max_frames=251):
    try:
        mel = librosa.feature.melspectrogram(
            y=y, sr=sr, n_fft=n_fft, hop_length=hop_length,
            n_mels=n_mels, fmin=fmin, fmax=fmax, power=2.0
        )
        log_mel = librosa.power_to_db(mel, ref=np.max).astype(np.float32)
        if log_mel.shape[1] >= max_frames:
            log_mel = log_mel[:, :max_frames]
        else:
            log_mel = np.pad(log_mel, ((0, 0), (0, max_frames - log_mel.shape[1])), mode="edge")
        mean = np.mean(log_mel)
        std = np.std(log_mel) + 1e-6
        return (log_mel - mean) / std
    except Exception:
        return np.zeros((n_mels, max_frames), dtype=np.float32)

test_row = train_df[train_df["key"] == "bonafide"].iloc[0]
raw_test = load_raw_audio(test_row["file_path"])
proc_test = preprocess_audio(raw_test, is_train=False)
mel_test = extract_log_mel(proc_test)

print(f"Raw Audio Length:        {len(raw_test):,} samples")
print(f"Processed Audio Length:  {len(proc_test):,} samples (Expected: 64,000)")
print(f"Log-Mel Spectrogram Dim: {mel_test.shape} (Expected: (128, 251))")
print(f"Standardized Mean:       {mel_test.mean():.6f} (Expected: ~0.0)")
print(f"Standardized Std:        {mel_test.std():.6f} (Expected: ~1.0)")
assert len(proc_test) == 64000
assert mel_test.shape == (128, 251)


In [ ]:
bon_sample_path = train_df[train_df["key"] == "bonafide"].iloc[0]["file_path"]
a01_sample_path = train_df[train_df["attack_id"] == "A01"].iloc[0]["file_path"]
a04_sample_path = train_df[train_df["attack_id"] == "A04"].iloc[0]["file_path"]

sig_bon = load_raw_audio(bon_sample_path)
sig_a01 = load_raw_audio(a01_sample_path)
sig_a04 = load_raw_audio(a04_sample_path)

fig, axes = plt.subplots(3, 2, figsize=(15, 8), sharex="col")
zoom_len = 1600

demo_signals = [
    (sig_bon, "Authentic Human Voice (VCTK Corpus)", "steelblue"),
    (sig_a01, "TTS: Neural Acoustic + WaveNet (A01)", "firebrick"),
    (sig_a04, "Voice Conversion: Pitch Shifting + STRAIGHT (A04)", "darkorange")
]

for row_idx, (sig, label, col) in enumerate(demo_signals):
    t_full = np.linspace(0, len(sig) / 16000, len(sig))
    axes[row_idx, 0].plot(t_full, sig, color=col, lw=0.6)
    axes[row_idx, 0].set_ylabel("Amplitude", fontsize=10)
    axes[row_idx, 0].set_title(f"Full Utterance Waveform: {label}", fontsize=11)
    axes[row_idx, 0].grid(True, linestyle="--", alpha=0.3)

    t_zoom = np.linspace(0, zoom_len / 16000, zoom_len)
    start_pt = min(int(16000 * 0.8), max(0, len(sig) - zoom_len))
    axes[row_idx, 1].plot(t_zoom * 1000, sig[start_pt:start_pt + zoom_len], color=col, lw=1.2)
    axes[row_idx, 1].set_ylabel("Amplitude", fontsize=10)
    axes[row_idx, 1].set_title(f"Zoomed Glottal Waveform (100ms): {label}", fontsize=11)
    axes[row_idx, 1].grid(True, linestyle="--", alpha=0.3)

axes[2, 0].set_xlabel("Time (Seconds)", fontsize=11)
axes[2, 1].set_xlabel("Time (Milliseconds)", fontsize=11)

plt.tight_layout()
plt.savefig(os.path.join(fig_dir, "04_waveform_time_domain_comparison.png"), dpi=300, bbox_inches="tight")
plt.show()
print("Saved Figure 04: Waveform and glottal pulse inspection.")


In [ ]:
from scipy import signal as scipy_signal

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

for sig, label, col in demo_signals:
    freqs, psd = scipy_signal.welch(sig, fs=16000, nperseg=1024)
    psd_db = 10 * np.log10(psd + 1e-12)
    axes[0].plot(freqs, psd_db, label=label, color=col, lw=1.5)

axes[0].set_title("Power Spectral Density (Before Preprocessing)", fontsize=12)
axes[0].set_xlabel("Frequency (Hz)", fontsize=11)
axes[0].set_ylabel("Power / Frequency (dB/Hz)", fontsize=11)
axes[0].legend(fontsize=9)
axes[0].grid(True, linestyle="--", alpha=0.3)
axes[0].set_xlim(0, 8000)

for sig, label, col in demo_signals:
    sig_proc = preprocess_audio(sig, is_train=False)
    freqs, psd = scipy_signal.welch(sig_proc, fs=16000, nperseg=1024)
    psd_db = 10 * np.log10(psd + 1e-12)
    axes[1].plot(freqs, psd_db, label=label, color=col, lw=1.5)

axes[1].set_title("Power Spectral Density (After Pre-emphasis & Normalization)", fontsize=12)
axes[1].set_xlabel("Frequency (Hz)", fontsize=11)
axes[1].set_ylabel("Normalized Power (dB/Hz)", fontsize=11)
axes[1].legend(fontsize=9)
axes[1].grid(True, linestyle="--", alpha=0.3)
axes[1].set_xlim(0, 8000)

plt.tight_layout()
plt.savefig(os.path.join(fig_dir, "05_power_spectral_density_frequency_rolloff.png"), dpi=300, bbox_inches="tight")
plt.show()
print("Saved Figure 05: Power spectral density and frequency rolloff.")


In [ ]:
mel_fb = librosa.filters.mel(sr=16000, n_fft=1024, n_mels=128, fmin=20, fmax=8000)
freq_hz = np.linspace(0, 8000, mel_fb.shape[1])

plt.figure(figsize=(11, 4.5))
for i in range(0, 128, 4):
    plt.plot(freq_hz, mel_fb[i], alpha=0.7, lw=1.2)

plt.title("Triangular Mel Filterbank Frequency Response (128 Channels, 20-8000 Hz)", fontsize=12)
plt.xlabel("Frequency (Hz)", fontsize=11)
plt.ylabel("Filter Weight Amplitude", fontsize=11)
plt.grid(True, linestyle="--", alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(fig_dir, "06_mel_filterbank_frequency_response.png"), dpi=300, bbox_inches="tight")
plt.show()
print("Saved Figure 06: Mel filterbank frequency response.")


In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True)

for idx, (sig, label, _) in enumerate(demo_signals):
    proc = preprocess_audio(sig, is_train=False)
    feat = extract_log_mel(proc)
    im = axes[idx].imshow(feat, origin="lower", aspect="auto", cmap="viridis", interpolation="nearest")
    axes[idx].set_title(f"128-Channel Standardized Log-Mel Spectrogram: {label}", fontsize=11)
    axes[idx].set_ylabel("Mel Frequency Bins", fontsize=10)
    fig.colorbar(im, ax=axes[idx], format="%+2.1f")

axes[2].set_xlabel("Time Frame Index (251 Frames @ 256 Hop Length)", fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(fig_dir, "07_log_mel_spectrogram_representations.png"), dpi=300, bbox_inches="tight")
plt.show()
print("Saved Figure 07: Log-Mel spectrogram representations across attacks.")


In [ ]:
class MelDataset(Dataset):
    def __init__(self, df, is_train=False):
        self.df = df.reset_index(drop=True)
        self.is_train = is_train

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        raw = load_raw_audio(row["file_path"])
        proc = preprocess_audio(raw, is_train=self.is_train)
        feat = extract_log_mel(proc)

        if self.is_train:
            if np.random.rand() < 0.5:
                w_t = np.random.randint(1, 31)
                t_0 = np.random.randint(0, max(1, 251 - w_t))
                feat[:, t_0:t_0 + w_t] = feat.mean()
            if np.random.rand() < 0.5:
                w_f = np.random.randint(1, 17)
                f_0 = np.random.randint(0, max(1, 128 - w_f))
                feat[f_0:f_0 + w_f, :] = feat.mean()

        tensor_x = torch.from_numpy(feat).unsqueeze(0)
        tensor_y = torch.tensor(int(row["is_spoof"]), dtype=torch.long)
        return tensor_x, tensor_y

train_dataset = MelDataset(train_df, is_train=True)
dev_dataset = MelDataset(dev_df, is_train=False)
eval_dataset = MelDataset(eval_df, is_train=False) if len(eval_df) > 0 else None

train_targets = train_df["is_spoof"].values
class_counts = np.bincount(train_targets)
class_weights = 1.0 / np.maximum(class_counts, 1)
sample_weights = torch.FloatTensor(class_weights[train_targets])
sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

batch_size = 64
num_workers = 2

train_loader = DataLoader(train_dataset, batch_size=batch_size, sampler=sampler, num_workers=num_workers, pin_memory=True)
dev_loader = DataLoader(dev_dataset, batch_size=batch_size * 2, shuffle=False, num_workers=num_workers, pin_memory=True)
eval_loader = DataLoader(eval_dataset, batch_size=batch_size * 2, shuffle=False, num_workers=num_workers, pin_memory=True) if eval_dataset else None

print(f"DataLoaders Constructed:")
print(f"  Train: {len(train_loader)} batches (Batch Size: {batch_size}, Weighted Balanced Sampling)")
print(f"  Dev:   {len(dev_loader)} batches (Batch Size: {batch_size * 2})")
if eval_loader:
    print(f"  Eval:  {len(eval_loader)} batches (Batch Size: {batch_size * 2}, Full 71,237 Utterances)")


In [ ]:
class SEBlock(nn.Module):
    def __init__(self, channels, reduction=8):
        super().__init__()
        self.fc = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(channels, channels // reduction),
            nn.ReLU(),
            nn.Linear(channels // reduction, channels),
            nn.Sigmoid()
        )

    def forward(self, x):
        w = self.fc(x).view(x.size(0), x.size(1), 1, 1)
        return x * w

class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, stride=stride, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(out_ch)
        )
        self.se = SEBlock(out_ch)
        if stride != 1 or in_ch != out_ch:
            self.skip = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_ch)
            )
        else:
            self.skip = nn.Identity()

    def forward(self, x):
        return F.relu(self.se(self.conv(x)) + self.skip(x))

class SEResNet18(nn.Module):
    def __init__(self, in_channels=1, num_classes=2, dropout=0.3):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(in_channels, 32, kernel_size=7, stride=2, padding=3, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        )
        self.layer1 = nn.Sequential(ResBlock(32, 32), ResBlock(32, 32))
        self.layer2 = nn.Sequential(ResBlock(32, 64, stride=2), ResBlock(64, 64))
        self.layer3 = nn.Sequential(ResBlock(64, 128, stride=2), ResBlock(128, 128))
        self.layer4 = nn.Sequential(ResBlock(128, 256, stride=2), ResBlock(256, 256))
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc_latent = nn.Linear(256, 64)
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(dropout),
            self.fc_latent,
            nn.ReLU(),
            nn.Linear(64, num_classes)
        )

    def extract_latent(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.pool(x)
        x = torch.flatten(x, 1)
        return self.fc_latent(x)

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.pool(x)
        return self.head(x)

model = SEResNet18().to(device)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total Trainable Parameters: {total_params:,}")

with torch.no_grad():
    dummy_input = torch.zeros(2, 1, 128, 251).to(device)
    dummy_out = model(dummy_input)
    dummy_latent = model.extract_latent(dummy_input)
    print(f"Logits Output Shape: {dummy_out.shape} (Expected: (2, 2))")
    print(f"Latent Output Shape: {dummy_latent.shape} (Expected: (2, 64))")


In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.75, gamma=2.0, label_smoothing=0.05):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.label_smoothing = label_smoothing

    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, reduction="none", label_smoothing=self.label_smoothing)
        p_t = torch.exp(-ce)
        alpha_t = torch.where(targets == 1, self.alpha, 1.0 - self.alpha)
        loss = alpha_t * ((1.0 - p_t) ** self.gamma) * ce
        return loss.mean()

criterion = FocalLoss(alpha=0.75, gamma=2.0, label_smoothing=0.05)
print("Focal Loss criterion configured with alpha=0.75, gamma=2.0, label_smoothing=0.05.")


In [ ]:
def calculate_biometrics(y_true, y_score):
    fpr, tpr, thresholds = roc_curve(y_true, y_score, pos_label=1)
    fnr = 1.0 - tpr
    diff = np.abs(fpr - fnr)
    idx = np.nanargmin(diff)
    eer = float((fpr[idx] + fnr[idx]) / 2.0)
    opt_threshold = float(np.clip(thresholds[idx], 0.0, 1.0))
    auc = float(roc_auc_score(y_true, y_score))

    c_miss = 1.0
    c_fa = 10.0
    p_tar = 0.05
    p_spoof = 0.90
    cm_cost = c_miss * p_tar * fnr + c_fa * p_spoof * fpr
    norm_factor = min(c_miss * p_tar, c_fa * p_spoof)
    min_tdcf = float(np.min(cm_cost) / norm_factor)

    return {
        "eer": eer,
        "optimal_threshold": opt_threshold,
        "auc": auc,
        "min_tdcf": min_tdcf,
        "fpr": fpr,
        "tpr": tpr,
        "fnr": fnr,
        "thresholds": thresholds
    }

def evaluate_network_batches(net, loader, compute_device, use_mixed_precision, log_interval=50):
    net.eval()
    prob_list = []
    target_list = []
    total_batches = len(loader)
    t_start = time.time()

    with torch.no_grad():
        for b_idx, (x_b, y_b) in enumerate(loader, 1):
            x_b = x_b.to(compute_device)
            with torch.amp.autocast(device_type=compute_device.type, enabled=use_mixed_precision):
                logits = net(x_b)
                probs = torch.softmax(logits, dim=1)[:, 1]
            prob_list.append(probs.cpu().numpy())
            target_list.append(y_b.numpy())

            if b_idx % log_interval == 0 or b_idx == total_batches:
                proc_cnt = min(b_idx * loader.batch_size, len(loader.dataset))
                elapsed = time.time() - t_start
                print(f"    [Batch {b_idx:03d}/{total_batches:03d}] Processed {proc_cnt:,} / {len(loader.dataset):,} utterances ({elapsed:.1f}s)")

    all_probs = np.concatenate(prob_list)
    all_targets = np.concatenate(target_list)
    metrics = calculate_biometrics(all_targets, all_probs)
    return metrics, all_probs, all_targets

print("Biometric evaluation engine and batch progress tracker initialized.")


In [ ]:
epochs = 20
lr_init = 5e-4
lr_min = 1e-6
weight_decay = 1e-4

optimizer = torch.optim.AdamW(model.parameters(), lr=lr_init, weight_decay=weight_decay)
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2, eta_min=lr_min)
scaler = torch.amp.GradScaler(device=device.type, enabled=use_amp)

best_model_path = os.path.join(weights_dir, "se_resnet18_best.pth")
history_file = os.path.join(weights_dir, "training_history.json")

best_dev_eer = float("inf")
best_dev_auc = 0.0
best_dev_threshold = 0.5
training_history = []

print(f"Commencing SE-ResNet-18 End-to-End Training ({epochs} Epochs)")
print("=" * 85)

for epoch in range(1, epochs + 1):
    t_start = time.time()
    model.train()
    running_loss = 0.0

    for x_b, y_b in train_loader:
        x_b = x_b.to(device)
        y_b = y_b.to(device)

        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast(device_type=device.type, enabled=use_amp):
            outputs = model(x_b)
            loss = criterion(outputs, y_b)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * x_b.size(0)

    epoch_loss = running_loss / len(train_dataset)
    scheduler.step()

    dev_metrics, _, _ = evaluate_network_batches(model, dev_loader, device, use_amp, log_interval=100)
    dev_eer = dev_metrics["eer"]
    dev_auc = dev_metrics["auc"]
    dev_thresh = dev_metrics["optimal_threshold"]
    dev_tdcf = dev_metrics["min_tdcf"]

    elapsed = time.time() - t_start
    log_entry = {
        "epoch": epoch,
        "train_loss": round(epoch_loss, 5),
        "dev_eer": round(dev_eer, 5),
        "dev_auc": round(dev_auc, 5),
        "dev_min_tdcf": round(dev_tdcf, 5),
        "dev_threshold": round(dev_thresh, 5),
        "time_seconds": round(elapsed, 1)
    }
    training_history.append(log_entry)

    is_optimal = dev_eer < best_dev_eer
    if is_optimal:
        best_dev_eer = dev_eer
        best_dev_auc = dev_auc
        best_dev_threshold = dev_thresh
        torch.save({
            "epoch": epoch,
            "state_dict": model.state_dict(),
            "eer": best_dev_eer,
            "auc": best_dev_auc,
            "threshold": best_dev_threshold,
            "architecture": "SE-ResNet-18"
        }, best_model_path)

    star = " [NEW BEST]" if is_optimal else ""
    print(f"Epoch [{epoch:02d}/{epochs:02d}] | Loss: {epoch_loss:.4f} | Dev EER: {dev_eer*100:.3f}% | Dev AUC: {dev_auc:.4f} | min t-DCF: {dev_tdcf:.4f} | Time: {elapsed:.0f}s{star}")

with open(history_file, "w", encoding="utf-8") as f:
    json.dump(training_history, f, indent=2)

print("=" * 85)
print(f"Training Complete. Optimal Dev EER: {best_dev_eer*100:.3f}% | Dev AUC: {best_dev_auc:.4f}")
print(f"Optimal Checkpoint Saved: {best_model_path}")


In [ ]:
epochs_x = [e["epoch"] for e in training_history]
losses_y = [e["train_loss"] for e in training_history]
eers_y = [e["dev_eer"] * 100 for e in training_history]
aucs_y = [e["dev_auc"] for e in training_history]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(epochs_x, losses_y, color="midnightblue", lw=2, marker="o", markersize=4, label="Training Loss (Focal)")
axes[0].set_title("Training Loss Convergence Across Epochs", fontsize=12)
axes[0].set_xlabel("Epoch Number", fontsize=11)
axes[0].set_ylabel("Loss Magnitude", fontsize=11)
axes[0].grid(True, linestyle="--", alpha=0.3)
axes[0].legend(fontsize=10)

axes[1].plot(epochs_x, eers_y, color="crimson", lw=2, marker="s", markersize=4, label="Development EER (%)")
axes[1].plot(epochs_x, [a * 100 for a in aucs_y], color="forestgreen", lw=1.5, linestyle="--", label="Development AUC (x100)")
axes[1].set_title("Biometric Validation Trajectory: Dev EER and AUC", fontsize=12)
axes[1].set_xlabel("Epoch Number", fontsize=11)
axes[1].set_ylabel("Percentage (%)", fontsize=11)
axes[1].grid(True, linestyle="--", alpha=0.3)
axes[1].legend(fontsize=10)

plt.tight_layout()
plt.savefig(os.path.join(fig_dir, "08_training_loss_and_eer_curves.png"), dpi=300, bbox_inches="tight")
plt.show()
print("Saved Figure 08: Training loss and biometric validation trajectory.")


In [ ]:
checkpoint = torch.load(best_model_path, map_location=device)
model.load_state_dict(checkpoint["state_dict"])
model.eval()
print(f"Loaded optimal checkpoint from epoch {checkpoint['epoch']} with Dev EER: {checkpoint['eer']*100:.3f}%")

print("Validating Development Partition with Loaded Optimal Checkpoint...")
dev_metrics, dev_scores, dev_targets = evaluate_network_batches(model, dev_loader, device, use_amp, log_interval=50)
optimal_threshold = dev_metrics["optimal_threshold"]

print("")
print("Executing Full-Scale Batch Inference on ASVspoof 2019 Evaluation Partition...")
print(f"Total Target Evaluation Utterances: {len(eval_df):,}")
t_eval_start = time.time()

eval_metrics, eval_scores, eval_targets = evaluate_network_batches(model, eval_loader, device, use_amp, log_interval=50)
t_eval_elapsed = time.time() - t_eval_start

print(f"Evaluation Partition Inference Completed in {t_eval_elapsed:.1f}s ({len(eval_df)/max(t_eval_elapsed, 1):.0f} utterances/sec)")
print("")
print("=" * 70)
print("ASVSPOOF 2019 COMPREHENSIVE BIOMETRIC PERFORMANCE BENCHMARK")
print("=" * 70)
print("Development Partition (Known Attacks A01 - A06):")
print(f"  Equal Error Rate (EER):      {dev_metrics['eer']*100:.3f}%")
print(f"  Normalized min t-DCF:        {dev_metrics['min_tdcf']:.4f}")
print(f"  Area Under ROC Curve (AUC):  {dev_metrics['auc']:.4f}")
print(f"  Calibrated Decision Thresh:  {optimal_threshold:.4f}")
print("")
print("Evaluation Partition (Unseen Out-of-Distribution Attacks A07 - A19):")
print(f"  Equal Error Rate (EER):      {eval_metrics['eer']*100:.3f}%")
print(f"  Normalized min t-DCF:        {eval_metrics['min_tdcf']:.4f}")
print(f"  Area Under ROC Curve (AUC):  {eval_metrics['auc']:.4f}")
print("=" * 70)


In [ ]:
plt.figure(figsize=(7, 6))
plt.plot(dev_metrics["fpr"], dev_metrics["tpr"], color="steelblue", lw=2, label=f"Dev Partition (AUC = {dev_metrics['auc']:.4f}, EER = {dev_metrics['eer']*100:.2f}%)")
plt.plot(eval_metrics["fpr"], eval_metrics["tpr"], color="crimson", lw=2, label=f"Eval Partition (AUC = {eval_metrics['auc']:.4f}, EER = {eval_metrics['eer']*100:.2f}%)")
plt.plot([0, 1], [0, 1], color="gray", linestyle=":", label="Random Classifier (AUC = 0.5000)")
plt.scatter([dev_metrics["eer"]], [1 - dev_metrics["eer"]], color="blue", s=60, zorder=5, label=f"Dev Operating Point ({dev_metrics['eer']*100:.2f}%)")
plt.scatter([eval_metrics["eer"]], [1 - eval_metrics["eer"]], color="darkred", s=60, zorder=5, label=f"Eval Operating Point ({eval_metrics['eer']*100:.2f}%)")

plt.title("Receiver Operating Characteristic (ROC) Benchmark: Dev vs Eval", fontsize=12)
plt.xlabel("False Positive Rate (FPR)", fontsize=11)
plt.ylabel("True Positive Rate (TPR)", fontsize=11)
plt.legend(loc="lower right", fontsize=9)
plt.grid(True, linestyle="--", alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(fig_dir, "09_receiver_operating_characteristic_roc.png"), dpi=300, bbox_inches="tight")
plt.show()
print("Saved Figure 09: ROC curves (Dev vs Eval).")


In [ ]:
from scipy.stats import norm

def plot_det(fpr, fnr, label, color, ax):
    fpr_clip = np.clip(fpr, 1e-5, 1 - 1e-5)
    fnr_clip = np.clip(fnr, 1e-5, 1 - 1e-5)
    ax.plot(norm.ppf(fpr_clip), norm.ppf(fnr_clip), label=label, color=color, lw=2)

fig, ax = plt.subplots(figsize=(7, 6))
plot_det(dev_metrics["fpr"], dev_metrics["fnr"], f"Dev Partition (EER = {dev_metrics['eer']*100:.2f}%)", "steelblue", ax)
plot_det(eval_metrics["fpr"], eval_metrics["fnr"], f"Eval Partition (EER = {eval_metrics['eer']*100:.2f}%)", "crimson", ax)

ticks = [0.001, 0.01, 0.05, 0.20, 0.50]
tick_locs = norm.ppf(ticks)
tick_labels = [f"{t*100:.1f}%" for t in ticks]
ax.set_xticks(tick_locs)
ax.set_xticklabels(tick_labels)
ax.set_yticks(tick_locs)
ax.set_yticklabels(tick_labels)

ax.set_title("Detection Error Tradeoff (DET) Benchmark: Dev vs Eval", fontsize=12)
ax.set_xlabel("False Alarm Rate / FPR (%)", fontsize=11)
ax.set_ylabel("Miss Rate / FNR (%)", fontsize=11)
ax.legend(loc="upper right", fontsize=10)
ax.grid(True, linestyle="--", alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(fig_dir, "10_detection_error_tradeoff_det.png"), dpi=300, bbox_inches="tight")
plt.show()
print("Saved Figure 10: DET curves (Dev vs Eval).")


In [ ]:
prec_dev, rec_dev, _ = precision_recall_curve(dev_targets, dev_scores)
prec_eval, rec_eval, _ = precision_recall_curve(eval_targets, eval_scores)
ap_dev = average_precision_score(dev_targets, dev_scores)
ap_eval = average_precision_score(eval_targets, eval_scores)

plt.figure(figsize=(7, 6))
plt.plot(rec_dev, prec_dev, color="steelblue", lw=2, label=f"Dev Partition (AP = {ap_dev:.4f})")
plt.plot(rec_eval, prec_eval, color="crimson", lw=2, label=f"Eval Partition (AP = {ap_eval:.4f})")
plt.title("Precision-Recall (PR) Curve Benchmark: Dev vs Eval", fontsize=12)
plt.xlabel("Recall (True Spoof Coverage)", fontsize=11)
plt.ylabel("Precision (Positive Predictive Value)", fontsize=11)
plt.legend(loc="lower left", fontsize=10)
plt.grid(True, linestyle="--", alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(fig_dir, "11_precision_recall_curves.png"), dpi=300, bbox_inches="tight")
plt.show()
print("Saved Figure 11: Precision-Recall curves (Dev vs Eval).")


In [ ]:
eval_preds_binary = (eval_scores >= optimal_threshold).astype(int)
cm = confusion_matrix(eval_targets, eval_preds_binary)
cm_norm = cm.astype("float") / cm.sum(axis=1)[:, np.newaxis]

plt.figure(figsize=(6, 5))
sns.heatmap(cm_norm, annot=True, fmt=".3f", cmap="Blues", cbar=True,
            xticklabels=["Authentic (Bonafide)", "Synthetic (Spoof)"],
            yticklabels=["Authentic (Bonafide)", "Synthetic (Spoof)"],
            annot_kws={"size": 12, "weight": "bold"})
plt.title(f"Normalized Confusion Matrix: Evaluation Partition (N={len(eval_targets):,})", fontsize=11)
plt.xlabel("Predicted Class Label", fontsize=10)
plt.ylabel("Ground Truth Class Label", fontsize=10)

tn, fp, fn, tp = cm.ravel()
acc_eval = (tp + tn) / len(eval_targets)
prec_eval_val = tp / max(tp + fp, 1)
rec_eval_val = tp / max(tp + fn, 1)
f1_eval_val = 2 * (prec_eval_val * rec_eval_val) / max(prec_eval_val + rec_eval_val, 1e-6)

print("Evaluation Confusion Matrix Performance:")
print(f"  True Negatives (Authentic Correct):    {tn:,} ({cm_norm[0,0]*100:.2f}%)")
print(f"  False Positives (Authentic Misclassed): {fp:,} ({cm_norm[0,1]*100:.2f}%)")
print(f"  False Negatives (Spoof Undetected):    {fn:,} ({cm_norm[1,0]*100:.2f}%)")
print(f"  True Positives (Spoof Detected):       {tp:,} ({cm_norm[1,1]*100:.2f}%)")
print(f"  Overall Evaluation Accuracy:           {acc_eval*100:.2f}%")
print(f"  Overall Evaluation F1-Score:           {f1_eval_val:.4f}")

plt.tight_layout()
plt.savefig(os.path.join(fig_dir, "12_normalized_confusion_matrix_eval.png"), dpi=300, bbox_inches="tight")
plt.show()
print("Saved Figure 12: Normalized confusion matrix on the Evaluation partition.")


In [ ]:
eval_df_scored = eval_df.copy()
eval_df_scored["spoof_score"] = eval_scores
eval_df_scored["pred_label"] = eval_preds_binary

dev_df_scored = dev_df.copy()
dev_df_scored["spoof_score"] = dev_scores
dev_preds_binary = (dev_scores >= optimal_threshold).astype(int)
dev_df_scored["pred_label"] = dev_preds_binary

attack_tech_mapping = {
    "A01": ("Known (Dev)", "TTS: Neural Acoustic (AR RNN) + WaveNet"),
    "A02": ("Known (Dev)", "TTS: Neural Acoustic (AR RNN) + WORLD"),
    "A03": ("Known (Dev)", "TTS: Concatenative Unit Selection"),
    "A04": ("Known (Dev & Eval)", "VC: Formant / Pitch Shifting + STRAIGHT"),
    "A05": ("Known (Dev)", "VC: Variational Autoencoder (VAE)"),
    "A06": ("Known (Dev & Eval)", "VC: Transfer Function Regression + WORLD"),
    "A07": ("Unseen (Eval)", "TTS: Neural Acoustic + Waveform Filtering"),
    "A08": ("Unseen (Eval)", "TTS: Neural Acoustic + Spectral Filtering"),
    "A09": ("Unseen (Eval)", "TTS: Poly-Phase Vocoder"),
    "A10": ("Unseen (Eval)", "TTS: Autoregressive Neural Vocoder (WaveNet)"),
    "A11": ("Unseen (Eval)", "TTS: Non-Autoregressive Waveform Synthesis"),
    "A12": ("Unseen (Eval)", "TTS: Neural Source-Filter (NSF)"),
    "A13": ("Unseen (Eval)", "VC: Differential Formant Synthesis"),
    "A14": ("Unseen (Eval)", "VC: Direct Waveform Modification"),
    "A15": ("Unseen (Eval)", "VC: Adaptive Waveform Filtering"),
    "A16": ("Unseen (Eval)", "VC: Spectral Envelope Transformation"),
    "A17": ("Unseen (Eval)", "VC: High-Order Non-Linear Phase Mapping"),
    "A18": ("Unseen (Eval)", "VC: Formant-Preserving Pitch Synchronous"),
    "A19": ("Unseen (Eval)", "VC: Multi-Speaker Variational Transfer")
}

breakdown_records = []

bon_sub = eval_df_scored[eval_df_scored["key"] == "bonafide"]
if len(bon_sub) > 0:
    bon_corr = int((bon_sub["pred_label"] == 0).sum())
    breakdown_records.append({
        "Attack ID": "Bonafide",
        "Partition": "Eval",
        "Category": "Human Speech",
        "Synthesis Technology": "VCTK Authentic Multi-Speaker Audio",
        "Total Utterances": len(bon_sub),
        "Detection Accuracy (%)": round(bon_corr / len(bon_sub) * 100, 2),
        "Mean Spoof Score": round(float(bon_sub["spoof_score"].mean()), 4)
    })

eval_attack_ids = sorted([a for a in eval_df_scored["attack_id"].unique() if a != "Bonafide"])
for atk in eval_attack_ids:
    atk_sub = eval_df_scored[eval_df_scored["attack_id"] == atk]
    corr = int((atk_sub["pred_label"] == 1).sum())
    cat, tech = attack_tech_mapping.get(atk, ("Unseen (Eval)", "Unspecified Generative Model"))
    breakdown_records.append({
        "Attack ID": atk,
        "Partition": "Eval",
        "Category": cat,
        "Synthesis Technology": tech,
        "Total Utterances": len(atk_sub),
        "Detection Accuracy (%)": round(corr / len(atk_sub) * 100, 2),
        "Mean Spoof Score": round(float(atk_sub["spoof_score"].mean()), 4)
    })

dev_only_atks = ["A01", "A02", "A03", "A05"]
for atk in dev_only_atks:
    sub = dev_df_scored[dev_df_scored["attack_id"] == atk]
    if len(sub) > 0:
        corr = int((sub["pred_label"] == 1).sum())
        cat, tech = attack_tech_mapping.get(atk, ("Known (Dev)", "Dev Generative Model"))
        breakdown_records.append({
            "Attack ID": atk,
            "Partition": "Dev",
            "Category": cat,
            "Synthesis Technology": tech,
            "Total Utterances": len(sub),
            "Detection Accuracy (%)": round(corr / len(sub) * 100, 2),
            "Mean Spoof Score": round(float(sub["spoof_score"].mean()), 4)
        })

breakdown_df = pd.DataFrame(breakdown_records)
print(breakdown_df.to_string(index=False))

csv_out_path = "/kaggle/working/attack_vulnerability_breakdown.csv"
breakdown_df.to_csv(csv_out_path, index=False)
print(f"Exported Attack Breakdown Table to {csv_out_path}")

eval_plot_records = [r for r in breakdown_records if r["Partition"] == "Eval" and r["Attack ID"] != "Bonafide"]
eval_plot_df = pd.DataFrame(eval_plot_records).sort_values("Attack ID")

plt.figure(figsize=(15, 6))
bar_colors = ["coral" if "Known" in cat else "mediumpurple" for cat in eval_plot_df["Category"]]
bars = plt.bar(eval_plot_df["Attack ID"], eval_plot_df["Detection Accuracy (%)"], color=bar_colors, edgecolor="black", width=0.6)
plt.axhline(100.0, color="gray", linestyle=":", lw=1)
plt.axhline(acc_eval * 100, color="crimson", linestyle="--", lw=1.5, label=f"Average Eval Accuracy ({acc_eval*100:.2f}%)")

plt.title("Attack-by-Attack Detection Accuracy on ASVspoof 2019 Evaluation Partition", fontsize=12)
plt.xlabel("Spoofing Algorithm Identifier: Known (Coral) vs Unseen OOD (Purple)", fontsize=11)
plt.ylabel("Detection Accuracy (%)", fontsize=11)
plt.ylim(0, 112)
plt.legend(loc="lower right", fontsize=10)
plt.grid(axis="y", linestyle="--", alpha=0.3)

for bar in bars:
    h = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, h + 1.2, f"{h:.1f}%", ha="center", va="bottom", fontsize=8, rotation=45)

plt.tight_layout()
plt.savefig(os.path.join(fig_dir, "13_attack_by_attack_accuracy_barchart.png"), dpi=300, bbox_inches="tight")
plt.show()
print("Saved Figure 13: Granular attack-by-attack detection accuracy bar chart.")


In [ ]:
bon_tsne = eval_df_scored[eval_df_scored["key"] == "bonafide"].sample(min(300, len(eval_df_scored[eval_df_scored["key"] == "bonafide"])), random_state=42)
known_tsne = eval_df_scored[eval_df_scored["attack_id"].isin(["A04", "A06"])].sample(min(300, len(eval_df_scored[eval_df_scored["attack_id"].isin(["A04", "A06"])])), random_state=42)
unseen_tsne = eval_df_scored[eval_df_scored["attack_id"].isin(["A07", "A10", "A12", "A17", "A19"])].sample(min(400, len(eval_df_scored[eval_df_scored["attack_id"].isin(["A07", "A10", "A12", "A17", "A19"])])), random_state=42)

tsne_manifest = pd.concat([bon_tsne, known_tsne, unseen_tsne]).reset_index(drop=True)
tsne_dataset = MelDataset(tsne_manifest, is_train=False)
tsne_loader = DataLoader(tsne_dataset, batch_size=64, shuffle=False)

latents_list = []
model.eval()
with torch.no_grad():
    for x_b, _ in tsne_loader:
        x_b = x_b.to(device)
        with torch.amp.autocast(device_type=device.type, enabled=use_amp):
            lat = model.extract_latent(x_b)
        latents_list.append(lat.cpu().numpy())

latent_matrix = np.concatenate(latents_list)
print(f"Extracted 64-Dimensional Latent Matrix: {latent_matrix.shape}")

tsne_model = TSNE(n_components=2, perplexity=35, random_state=42, n_iter=1000)
coords_2d = tsne_model.fit_transform(latent_matrix)

plt.figure(figsize=(9, 7))
is_bon = tsne_manifest["key"] == "bonafide"
is_known = tsne_manifest["attack_id"].isin(["A04", "A06"])
is_unseen = ~is_bon & ~is_known

plt.scatter(coords_2d[is_bon, 0], coords_2d[is_bon, 1], color="steelblue", alpha=0.8, s=40, label="Authentic Human Voice (Bonafide)")
plt.scatter(coords_2d[is_known, 0], coords_2d[is_known, 1], color="forestgreen", alpha=0.8, s=40, label="Known Attacks (A04, A06)")
plt.scatter(coords_2d[is_unseen, 0], coords_2d[is_unseen, 1], color="crimson", alpha=0.8, s=40, label="Unseen OOD Attacks (A07, A10, A12, A17, A19)")

plt.title("t-SNE 2D Projection of SE-ResNet-18 64-Dimensional Latent Manifold", fontsize=12)
plt.xlabel("t-SNE Dimension 1", fontsize=11)
plt.ylabel("t-SNE Dimension 2", fontsize=11)
plt.legend(loc="best", fontsize=10)
plt.grid(True, linestyle="--", alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(fig_dir, "14_tsne_latent_clusters.png"), dpi=300, bbox_inches="tight")
plt.show()
print("Saved Figure 14: t-SNE 2D latent manifold clustering.")


In [ ]:
class GradCAMMel:
    def __init__(self, target_model, target_layer):
        self.model = target_model
        self.layer = target_layer
        self.activations = None
        self.gradients = None
        self.h1 = self.layer.register_forward_hook(self.forward_hook)
        self.h2 = self.layer.register_full_backward_hook(self.backward_hook)

    def forward_hook(self, module, inp, out):
        self.activations = out.detach()

    def backward_hook(self, module, grad_in, grad_out):
        self.gradients = grad_out[0].detach()

    def generate(self, input_tensor, target_class=1):
        self.model.eval()
        self.model.zero_grad()
        output = self.model(input_tensor)
        score = output[0, target_class]
        score.backward()

        weights = torch.mean(self.gradients, dim=(2, 3), keepdim=True)
        cam = torch.sum(weights * self.activations, dim=1, keepdim=True)
        cam = F.relu(cam)
        cam = F.interpolate(cam, size=(128, 251), mode="bilinear", align_corners=False)
        cam = cam.squeeze().cpu().numpy()
        cam_norm = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam_norm

    def cleanup(self):
        self.h1.remove()
        self.h2.remove()

cam_engine = GradCAMMel(model, model.layer4[1].conv[3])

bon_file = eval_df_scored[eval_df_scored["key"] == "bonafide"].iloc[0]["file_path"]
spf_file = eval_df_scored[eval_df_scored["attack_id"] == "A10"].iloc[0]["file_path"]

fig, axes = plt.subplots(2, 2, figsize=(15, 7))

for idx, (fpath, title, target_cls) in enumerate([
    (bon_file, "Authentic Speech (Bonafide)", 0),
    (spf_file, "Synthetic Speech (WaveNet A10)", 1)
]):
    raw = load_raw_audio(fpath)
    proc = preprocess_audio(raw, is_train=False)
    mel = extract_log_mel(proc)
    tens = torch.from_numpy(mel).unsqueeze(0).unsqueeze(0).to(device)

    saliency = cam_engine.generate(tens, target_class=target_cls)

    im0 = axes[idx, 0].imshow(mel, origin="lower", aspect="auto", cmap="viridis")
    axes[idx, 0].set_title(f"{title}: Input Standardized Log-Mel Spectrogram", fontsize=10)
    axes[idx, 0].set_ylabel("Mel Bins (128)", fontsize=9)
    fig.colorbar(im0, ax=axes[idx, 0], fraction=0.046, pad=0.04)

    axes[idx, 1].imshow(mel, origin="lower", aspect="auto", cmap="gray")
    im1 = axes[idx, 1].imshow(saliency, origin="lower", aspect="auto", cmap="jet", alpha=0.6)
    axes[idx, 1].set_title(f"{title}: Grad-CAM Artifact Saliency Heatmap", fontsize=10)
    axes[idx, 1].set_ylabel("Mel Bins (128)", fontsize=9)
    fig.colorbar(im1, ax=axes[idx, 1], fraction=0.046, pad=0.04)

axes[1, 0].set_xlabel("Time Frames (251)", fontsize=10)
axes[1, 1].set_xlabel("Time Frames (251)", fontsize=10)

plt.tight_layout()
plt.savefig(os.path.join(fig_dir, "15_gradcam_spectro_temporal_explainability.png"), dpi=300, bbox_inches="tight")
plt.show()
cam_engine.cleanup()
print("Saved Figure 15: Grad-CAM spectro-temporal explainability heatmaps.")


In [ ]:
def predict_audio_sample(file_path, net, decision_threshold):
    net.eval()
    raw = load_raw_audio(file_path)
    proc = preprocess_audio(raw, is_train=False)
    feat = extract_log_mel(proc)
    tensor_in = torch.from_numpy(feat).unsqueeze(0).unsqueeze(0).to(device)

    with torch.no_grad():
        with torch.amp.autocast(device_type=device.type, enabled=use_amp):
            prob = torch.softmax(net(tensor_in), dim=1)[0, 1].item()

    is_detected_spoof = prob >= decision_threshold
    decision = "SPOOF (SYNTHETIC VOICE DETECTED)" if is_detected_spoof else "BONAFIDE (AUTHENTIC HUMAN VOICE)"
    confidence = prob if is_detected_spoof else (1.0 - prob)

    return {
        "file_name": os.path.basename(file_path),
        "decision": decision,
        "spoof_probability": round(prob, 5),
        "confidence": f"{confidence * 100:.2f}%",
        "operating_threshold": round(decision_threshold, 4)
    }

demo_bon_path = eval_df_scored[eval_df_scored["key"] == "bonafide"].iloc[1]["file_path"]
demo_spf_path = eval_df_scored[eval_df_scored["attack_id"] == "A12"].iloc[1]["file_path"]

print("Live File-Level Biometric Inference Demonstration:")
print("")
print("--- Test Sample 1: Ground Truth Authentic ---")
res_bon = predict_audio_sample(demo_bon_path, model, optimal_threshold)
print(json.dumps(res_bon, indent=2))

print("")
print("--- Test Sample 2: Ground Truth Deepfake (A12 Neural Source-Filter) ---")
res_spf = predict_audio_sample(demo_spf_path, model, optimal_threshold)
print(json.dumps(res_spf, indent=2))

fig, ax = plt.subplots(figsize=(8, 3))
test_names = ["Sample 1 (Authentic)", "Sample 2 (NSF Deepfake A12)"]
test_probs = [res_bon["spoof_probability"], res_spf["spoof_probability"]]
test_colors = ["steelblue", "crimson"]

bars = ax.barh(test_names, test_probs, color=test_colors, height=0.4)
ax.axvline(optimal_threshold, color="black", linestyle="--", lw=1.5, label=f"Calibrated Threshold ({optimal_threshold:.4f})")
ax.set_xlim(0, 1.0)
ax.set_xlabel("Spoof Posterior Probability", fontsize=11)
ax.set_title("Single-File Live Inference Confidence Benchmark", fontsize=12)
ax.legend(loc="lower right", fontsize=10)
ax.grid(axis="x", linestyle="--", alpha=0.3)

for bar in bars:
    w = bar.get_width()
    ax.text(w + 0.02, bar.get_y() + bar.get_height()/2.0, f"{w:.4f}", va="center", fontsize=10, fontweight="bold")

plt.tight_layout()
plt.savefig(os.path.join(fig_dir, "16_single_file_inference_verification.png"), dpi=300, bbox_inches="tight")
plt.show()
print("Saved Figure 16: Single-file live inference verification.")


In [ ]:
final_summary_report = {
    "study_metadata": {
        "model_architecture": "SE-ResNet-18 (Squeeze-and-Excitation ResNet)",
        "front_end_features": "128-channel Log-Mel Spectrogram (Pre-emphasis + Standardization)",
        "input_tensor_shape": [1, 128, 251],
        "dataset_benchmark": "ASVspoof 2019 Logical Access",
        "training_epochs": epochs,
        "batch_size": batch_size,
        "loss_function": "Focal Loss (alpha=0.75, gamma=2.0, label_smoothing=0.05)",
        "optimizer": "AdamW (lr=5e-4, weight_decay=1e-4) with Cosine Annealing"
    },
    "development_partition_results": {
        "eer_percent": round(dev_metrics["eer"] * 100, 3),
        "normalized_min_tdcf": round(dev_metrics["min_tdcf"], 4),
        "roc_auc": round(dev_metrics["auc"], 4),
        "calibrated_decision_threshold": round(optimal_threshold, 4),
        "total_utterances": len(dev_targets)
    },
    "evaluation_partition_results": {
        "eer_percent": round(eval_metrics["eer"] * 100, 3),
        "normalized_min_tdcf": round(eval_metrics["min_tdcf"], 4),
        "roc_auc": round(eval_metrics["auc"], 4),
        "overall_accuracy_percent": round(acc_eval * 100, 2),
        "overall_f1_score": round(f1_eval_val, 4),
        "total_utterances": len(eval_targets)
    }
}

report_json_path = "/kaggle/working/experiment_final_report.json"
with open(report_json_path, "w", encoding="utf-8") as f:
    json.dump(final_summary_report, f, indent=2)

print("")
print("=" * 80)
print("FINAL ARTIFACT INVENTORY AND VERIFICATION")
print("=" * 80)
print(f"1. Checkpoint:         {best_model_path}")
print(f"2. Training History:   {history_file}")
print(f"3. Final Report JSON:  {report_json_path}")
print(f"4. Attack Breakdown:   {csv_out_path}")
print("5. Diagnostic Figures in /kaggle/working/figures/:")
for fig_file in sorted(os.listdir(fig_dir)):
    sz = os.path.getsize(os.path.join(fig_dir, fig_file))
    print(f"   - {fig_file} ({sz/1024:.1f} KB)")
print("=" * 80)
print("End-to-End research study completed successfully.")
